## Exporatory data analysis of the transaction and sales data

In [211]:
import sys
from pathlib import Path
import pandas as pd
from pandas import DataFrame
ROOT_PATH = str(Path.cwd().resolve().parent)
if ROOT_PATH not in sys.path:
    sys.path.append(ROOT_PATH)

In [212]:
ROOT_PATH

'/Users/taofiq/Documents/taofiq/plexure_data_engineering'

In [213]:
STORES_DATA_PATH = r'../data/raw/stores.csv'
TRANSACTION_DATA_PATH = r'../data/raw/sales_transactions.json'
DAILY_SUMMARY_REPORT_PATH = r'../data/results/daily_sumarry.csv'

In [214]:
df_sales = pd.read_csv(STORES_DATA_PATH)
df_txn = pd.read_json(TRANSACTION_DATA_PATH)

In [215]:
df_sales.head()

,store_id,store_name,region
0,akl-001,Auckland Central,Auckland
1,akl-002,Auckland Newmarket,Auckland
2,wlg-001,Wellington Lambton Quay,Wellington
3,chc-001,Christchurch Riccarton,Canterbury
4,dun-001,Dunedin George Street,Otago


In [216]:
len(df_sales)

5

In [217]:
df_txn.head(20)

,transaction_id,store_id,timestamp,customer_id,net_amount,store
0,tx-1001,akl-001,2026-04-01T10:00:00+13:00,cust-001,115.0,NaN
1,tx-1007,akl-001,2026-13-01T10:00:00+13:00,cust-006,34.5,NaN
2,tx-1003,wlg-001,2026-04-01T09:30:00+13:00,cust-003,92.0,NaN
3,tx-1011,NaN,2026-04-02T12:00:00+13:00,cust-010,69.0,chc-001
4,tx-1002,akl-001,2026-04-01T12:00:00+13:00,cust-002,57.5,NaN
5,tx-1008,unknown-999,2026-04-02T10:00:00+13:00,cust-007,23.0,NaN
6,tx-1004,chc-001,2026-04-02T11:00:00+13:00,cust-004,46.0,NaN
7,tx-1010,wlg-001,2026-04-02T18:00:00+13:00,cust-009,twenty,NaN
8,tx-1005,wlg-001,2026-04-02T14:30:00+13:00,cust-003,34.5,NaN
9,,akl-001,2026-04-02T15:30:00+13:00,cust-011,11.5,NaN


In [218]:
len(df_txn)

13

In [219]:
#small dataset, enforce col-types
COL_TYPES = {"transaction_id":'str',	"store_id":'str',	"timestamp":'timestamp',	"customer_id":'str',	"net_amount":'float',	"store":'str'}
# enforce column types according to COL_TYPES
df_txn = df_txn.copy()
## used AI to create the below
for col, target in COL_TYPES.items():
    if col not in df_txn.columns:
        continue
    if target == "str":
        df_txn[col] = df_txn[col].astype("string")
    elif target == "float":
        df_txn[col] = pd.to_numeric(df_txn[col], errors="coerce").astype("Float64")
    elif target == "timestamp":
        df_txn[col] = pd.to_datetime(df_txn[col], utc=False,errors="coerce")
    else:
        # fallback: try pandas astype with the provided string
        try:
            df_txn[col] = df_txn[col].astype(target)
        except Exception:
            pass

df_txn.dtypes

transaction_id                       string
store_id                             string
timestamp         datetime64[us, UTC+13:00]
customer_id                          string
net_amount                          Float64
store                                string
dtype: object

## Validating Business Rules

In [220]:
## add a rejection reason column to the transactoin data

df_txn_r = df_txn.copy()
df_txn_r['_rejection_reason'] = ""
df_txn_r

,transaction_id,store_id,timestamp,customer_id,net_amount,store,_rejection_reason
0,tx-1001,akl-001,2026-04-01 10:00:00+13:00,cust-001,115.0,<NA>,
1,tx-1007,akl-001,NaT,cust-006,34.5,<NA>,
2,tx-1003,wlg-001,2026-04-01 09:30:00+13:00,cust-003,92.0,<NA>,
3,tx-1011,<NA>,2026-04-02 12:00:00+13:00,cust-010,69.0,chc-001,
4,tx-1002,akl-001,2026-04-01 12:00:00+13:00,cust-002,57.5,<NA>,
5,tx-1008,unknown-999,2026-04-02 10:00:00+13:00,cust-007,23.0,<NA>,
6,tx-1004,chc-001,2026-04-02 11:00:00+13:00,cust-004,46.0,<NA>,
7,tx-1010,wlg-001,2026-04-02 18:00:00+13:00,cust-009,<NA>,<NA>,
8,tx-1005,wlg-001,2026-04-02 14:30:00+13:00,cust-003,34.5,<NA>,
9,,akl-001,2026-04-02 15:30:00+13:00,cust-011,11.5,<NA>,


In [221]:
# all amounts are NZD = numeric
## we cannot have nan/string/''/ etc.

df_txn_r['_rejection_reason'].eq("") & pd.to_numeric(df_txn_r['net_amount'],errors='coerce').isna()



0     False
1     False
2     False
3     False
4     False
5     False
6     False
7      True
8     False
9     False
10    False
11    False
12     True
dtype: bool

In [222]:

#slice invalid rows
df_txn_r.loc[df_txn_r['_rejection_reason'].eq("") & pd.to_numeric(df_txn_r['net_amount'],errors='coerce').isna()]




,transaction_id,store_id,timestamp,customer_id,net_amount,store,_rejection_reason
7,tx-1010,wlg-001,2026-04-02 18:00:00+13:00,cust-009,<NA>,<NA>,
12,tx-1009,akl-002,2026-04-02 15:00:00+13:00,cust-008,<NA>,<NA>,


In [223]:
## turn the above to a function to return a df
## add a column to state the error type if any

def validate_numeric_transactions(data:DataFrame, col:str)->DataFrame:
    """
    Convert the net_amount column to numeric and give reasons for rejection
    """
    df = data.copy()
    non_numeric_mask = (df['_rejection_reason'].eq("") & pd.to_numeric(df[col],errors='coerce').isna())
    df.loc[non_numeric_mask,"_rejection_reason"] = "non_numeric_amount"
    return df




In [224]:
test = validate_numeric_transactions(df_txn_r,'net_amount')
len(test)

13

In [225]:
test

,transaction_id,store_id,timestamp,customer_id,net_amount,store,_rejection_reason
0,tx-1001,akl-001,2026-04-01 10:00:00+13:00,cust-001,115.0,<NA>,
1,tx-1007,akl-001,NaT,cust-006,34.5,<NA>,
2,tx-1003,wlg-001,2026-04-01 09:30:00+13:00,cust-003,92.0,<NA>,
3,tx-1011,<NA>,2026-04-02 12:00:00+13:00,cust-010,69.0,chc-001,
4,tx-1002,akl-001,2026-04-01 12:00:00+13:00,cust-002,57.5,<NA>,
5,tx-1008,unknown-999,2026-04-02 10:00:00+13:00,cust-007,23.0,<NA>,
6,tx-1004,chc-001,2026-04-02 11:00:00+13:00,cust-004,46.0,<NA>,
7,tx-1010,wlg-001,2026-04-02 18:00:00+13:00,cust-009,<NA>,<NA>,non_numeric_amount
8,tx-1005,wlg-001,2026-04-02 14:30:00+13:00,cust-003,34.5,<NA>,
9,,akl-001,2026-04-02 15:30:00+13:00,cust-011,11.5,<NA>,


In [226]:
#Only include transactions that can be matched to a known store
valid_store_id = list(set(df_sales['store_id']))


df_txn_r.loc[df_txn_r['_rejection_reason'].eq("") & (~df_txn_r['store_id'].isin(valid_store_id))] 


,transaction_id,store_id,timestamp,customer_id,net_amount,store,_rejection_reason
3,tx-1011,<NA>,2026-04-02 12:00:00+13:00,cust-010,69.0,chc-001,
5,tx-1008,unknown-999,2026-04-02 10:00:00+13:00,cust-007,23.0,<NA>,


In [227]:
valid_store_id

['dun-001', 'akl-001', 'chc-001', 'akl-002', 'wlg-001']

In [228]:
from typing import List

def validate_store_id(data: DataFrame, stored_id: List[str], col: str) -> DataFrame:
    """
    Returns a dataframe with rejection reasons for the store id column:
    1. Missing store id
    2. Invalid store id
    """
    df = data.copy()

    missing_mask = df["_rejection_reason"].eq("") & (
        df[col].isna() | df[col].astype(str).str.strip().eq("")
    )
    invalid_mask = df["_rejection_reason"].eq("") & (~df[col].isin(stored_id)) & (~missing_mask)

    df.loc[missing_mask, "_rejection_reason"] = "missing_store"
    df.loc[invalid_mask, "_rejection_reason"] = "invalid_store_id"
    return df

In [229]:
t2 = validate_store_id(df_txn_r,valid_store_id,'store_id')
t2

,transaction_id,store_id,timestamp,customer_id,net_amount,store,_rejection_reason
0,tx-1001,akl-001,2026-04-01 10:00:00+13:00,cust-001,115.0,<NA>,
1,tx-1007,akl-001,NaT,cust-006,34.5,<NA>,
2,tx-1003,wlg-001,2026-04-01 09:30:00+13:00,cust-003,92.0,<NA>,
3,tx-1011,<NA>,2026-04-02 12:00:00+13:00,cust-010,69.0,chc-001,missing_store
4,tx-1002,akl-001,2026-04-01 12:00:00+13:00,cust-002,57.5,<NA>,
5,tx-1008,unknown-999,2026-04-02 10:00:00+13:00,cust-007,23.0,<NA>,invalid_store_id
6,tx-1004,chc-001,2026-04-02 11:00:00+13:00,cust-004,46.0,<NA>,
7,tx-1010,wlg-001,2026-04-02 18:00:00+13:00,cust-009,<NA>,<NA>,
8,tx-1005,wlg-001,2026-04-02 14:30:00+13:00,cust-003,34.5,<NA>,
9,,akl-001,2026-04-02 15:30:00+13:00,cust-011,11.5,<NA>,


In [230]:
##Deduplicate transactions using transaction_id


"""
There are two main issues with the transaction id

1. Missing transactions
2. Duplicated transactions

Address one first and then two in the same function so that clearly identify the reasons.
"""
def validate_transaction_id(data:DataFrame,col:str)->DataFrame:

    df = data.copy()
    empty_mask = df[col].astype(str).str.strip().eq("")
    duplicate_mask = df[col].duplicated(keep='first')
    df.loc[empty_mask,'_rejection_reason'] = "missing_transaction_id"
    df.loc[duplicate_mask,'_rejection_reason'] = "duplicate_transaction_id"
    return df


In [231]:
t3 = validate_transaction_id(df_txn_r,'transaction_id')
t3

,transaction_id,store_id,timestamp,customer_id,net_amount,store,_rejection_reason
0,tx-1001,akl-001,2026-04-01 10:00:00+13:00,cust-001,115.0,<NA>,
1,tx-1007,akl-001,NaT,cust-006,34.5,<NA>,
2,tx-1003,wlg-001,2026-04-01 09:30:00+13:00,cust-003,92.0,<NA>,
3,tx-1011,<NA>,2026-04-02 12:00:00+13:00,cust-010,69.0,chc-001,
4,tx-1002,akl-001,2026-04-01 12:00:00+13:00,cust-002,57.5,<NA>,
5,tx-1008,unknown-999,2026-04-02 10:00:00+13:00,cust-007,23.0,<NA>,
6,tx-1004,chc-001,2026-04-02 11:00:00+13:00,cust-004,46.0,<NA>,
7,tx-1010,wlg-001,2026-04-02 18:00:00+13:00,cust-009,<NA>,<NA>,
8,tx-1005,wlg-001,2026-04-02 14:30:00+13:00,cust-003,34.5,<NA>,
9,,akl-001,2026-04-02 15:30:00+13:00,cust-011,11.5,<NA>,missing_transaction_id


In [232]:
# create a validation for time also

def validate_timestamp(data:DataFrame, col:str)->DataFrame:
    df = data.copy()
    timestamp_mask = (df[col].isna() & df['_rejection_reason'].eq(""))
    df.loc[timestamp_mask,'_rejection_reason'] = 'invalid_timestamp'
    return df

In [233]:
t4 = validate_timestamp(df_txn_r,'timestamp')
t4

,transaction_id,store_id,timestamp,customer_id,net_amount,store,_rejection_reason
0,tx-1001,akl-001,2026-04-01 10:00:00+13:00,cust-001,115.0,<NA>,
1,tx-1007,akl-001,NaT,cust-006,34.5,<NA>,invalid_timestamp
2,tx-1003,wlg-001,2026-04-01 09:30:00+13:00,cust-003,92.0,<NA>,
3,tx-1011,<NA>,2026-04-02 12:00:00+13:00,cust-010,69.0,chc-001,
4,tx-1002,akl-001,2026-04-01 12:00:00+13:00,cust-002,57.5,<NA>,
5,tx-1008,unknown-999,2026-04-02 10:00:00+13:00,cust-007,23.0,<NA>,
6,tx-1004,chc-001,2026-04-02 11:00:00+13:00,cust-004,46.0,<NA>,
7,tx-1010,wlg-001,2026-04-02 18:00:00+13:00,cust-009,<NA>,<NA>,
8,tx-1005,wlg-001,2026-04-02 14:30:00+13:00,cust-003,34.5,<NA>,
9,,akl-001,2026-04-02 15:30:00+13:00,cust-011,11.5,<NA>,


In [234]:
## chain the pipeline

def run_pipeline(data:DataFrame)->DataFrame:
    df = data.copy()
    df = validate_transaction_id(data=df,col='transaction_id')
    df = validate_store_id(data=df,col='store_id',stored_id=valid_store_id)
    df = validate_timestamp(data=df,col='timestamp')
    df = validate_numeric_transactions(data=df,col='net_amount')
    return df

filtered_df = run_pipeline(df_txn_r)
filtered_df
    

,transaction_id,store_id,timestamp,customer_id,net_amount,store,_rejection_reason
0,tx-1001,akl-001,2026-04-01 10:00:00+13:00,cust-001,115.0,<NA>,
1,tx-1007,akl-001,NaT,cust-006,34.5,<NA>,invalid_timestamp
2,tx-1003,wlg-001,2026-04-01 09:30:00+13:00,cust-003,92.0,<NA>,
3,tx-1011,<NA>,2026-04-02 12:00:00+13:00,cust-010,69.0,chc-001,missing_store
4,tx-1002,akl-001,2026-04-01 12:00:00+13:00,cust-002,57.5,<NA>,
5,tx-1008,unknown-999,2026-04-02 10:00:00+13:00,cust-007,23.0,<NA>,invalid_store_id
6,tx-1004,chc-001,2026-04-02 11:00:00+13:00,cust-004,46.0,<NA>,
7,tx-1010,wlg-001,2026-04-02 18:00:00+13:00,cust-009,<NA>,<NA>,non_numeric_amount
8,tx-1005,wlg-001,2026-04-02 14:30:00+13:00,cust-003,34.5,<NA>,
9,,akl-001,2026-04-02 15:30:00+13:00,cust-011,11.5,<NA>,missing_transaction_id


In [235]:
valid = filtered_df[filtered_df['_rejection_reason'].eq("").copy()]
valid

,transaction_id,store_id,timestamp,customer_id,net_amount,store,_rejection_reason
0,tx-1001,akl-001,2026-04-01 10:00:00+13:00,cust-001,115.0,<NA>,
2,tx-1003,wlg-001,2026-04-01 09:30:00+13:00,cust-003,92.0,<NA>,
4,tx-1002,akl-001,2026-04-01 12:00:00+13:00,cust-002,57.5,<NA>,
6,tx-1004,chc-001,2026-04-02 11:00:00+13:00,cust-004,46.0,<NA>,
8,tx-1005,wlg-001,2026-04-02 14:30:00+13:00,cust-003,34.5,<NA>,
10,tx-1006,wlg-001,2026-04-02 16:10:00+13:00,cust-005,138.0,<NA>,


In [236]:
##unique_customers means distinct customer_id values per store per day
# add a date col
valid["date"] = pd.to_datetime(valid["timestamp"], errors="coerce").dt.date
valid.sort_values('store_id')

,transaction_id,store_id,timestamp,customer_id,net_amount,store,_rejection_reason,date
0,tx-1001,akl-001,2026-04-01 10:00:00+13:00,cust-001,115.0,<NA>,,2026-04-01
4,tx-1002,akl-001,2026-04-01 12:00:00+13:00,cust-002,57.5,<NA>,,2026-04-01
6,tx-1004,chc-001,2026-04-02 11:00:00+13:00,cust-004,46.0,<NA>,,2026-04-02
2,tx-1003,wlg-001,2026-04-01 09:30:00+13:00,cust-003,92.0,<NA>,,2026-04-01
8,tx-1005,wlg-001,2026-04-02 14:30:00+13:00,cust-003,34.5,<NA>,,2026-04-02
10,tx-1006,wlg-001,2026-04-02 16:10:00+13:00,cust-005,138.0,<NA>,,2026-04-02


In [237]:
df_agg = valid.\
    groupby(['date','store_id'])\
    .agg(
            total_sales_nzd   =("net_amount",   "sum"),
            unique_customers  =("customer_id",  "nunique"),
            average_invoice_nzd=("net_amount",  "mean"),
            max_invoice_nzd   =("net_amount",   "max"),
            min_invoice_nzd   =("net_amount",   "min"),
        ).reset_index()
df_agg

,date,store_id,total_sales_nzd,unique_customers,average_invoice_nzd,max_invoice_nzd,min_invoice_nzd
0,2026-04-01,akl-001,172.5,2,86.25,115.0,57.5
1,2026-04-01,wlg-001,92.0,1,92.0,92.0,92.0
2,2026-04-02,chc-001,46.0,1,46.0,46.0,46.0
3,2026-04-02,wlg-001,172.5,2,86.25,138.0,34.5


In [238]:
df_fact = df_agg.merge(df_sales[['store_id','store_name']], on='store_id', how='left').sort_values(["date", "store_id"])
df_fact[["date", "store_id", "store_name","total_sales_nzd", "unique_customers","average_invoice_nzd", "max_invoice_nzd", "min_invoice_nzd"]]

,date,store_id,store_name,total_sales_nzd,unique_customers,average_invoice_nzd,max_invoice_nzd,min_invoice_nzd
0,2026-04-01,akl-001,Auckland Central,172.5,2,86.25,115.0,57.5
1,2026-04-01,wlg-001,Wellington Lambton Quay,92.0,1,92.0,92.0,92.0
2,2026-04-02,chc-001,Christchurch Riccarton,46.0,1,46.0,46.0,46.0
3,2026-04-02,wlg-001,Wellington Lambton Quay,172.5,2,86.25,138.0,34.5


In [239]:
def aggregate_data(transaction_data:DataFrame, sales_data:DataFrame)->DataFrame:
    transaction_data = transaction_data.copy()
    sales_data = sales_data.copy()
    #filter the valid transactions
    valid = transaction_data[transaction_data['_rejection_reason'].eq("").copy()]
    valid["date"] = pd.to_datetime(valid["timestamp"], errors="coerce").dt.date
    valid.sort_values('store_id')

    #groupby and agg
    df_agg = valid.\
    groupby(['date','store_id'])\
    .agg(
            total_sales_nzd   =("net_amount",   "sum"),
            unique_customers  =("customer_id",  "nunique"),
            average_invoice_nzd=("net_amount",  "mean"),
            max_invoice_nzd   =("net_amount",   "max"),
            min_invoice_nzd   =("net_amount",   "min"),
        ).reset_index()
    #merge on 'sales_id'
    df_fact = df_agg.merge(sales_data[['store_id','store_name']], on='store_id', how='left').sort_values(["date", "store_id"])
    df_fact[["date", "store_id", "store_name","total_sales_nzd", "unique_customers","average_invoice_nzd", "max_invoice_nzd", "min_invoice_nzd"]]
    df_fact.to_csv(DAILY_SUMMARY_REPORT_PATH)
    return df_fact

In [240]:
final_df = aggregate_data(transaction_data=filtered_df,sales_data=df_sales)
final_df

,date,store_id,total_sales_nzd,unique_customers,average_invoice_nzd,max_invoice_nzd,min_invoice_nzd,store_name
0,2026-04-01,akl-001,172.5,2,86.25,115.0,57.5,Auckland Central
1,2026-04-01,wlg-001,92.0,1,92.0,92.0,92.0,Wellington Lambton Quay
2,2026-04-02,chc-001,46.0,1,46.0,46.0,46.0,Christchurch Riccarton
3,2026-04-02,wlg-001,172.5,2,86.25,138.0,34.5,Wellington Lambton Quay


In [241]:
valid.groupby(["date", "store_id"]).agg(
            total_sales_nzd   =("net_amount",   "sum"),
            unique_customers  =("customer_id",  "nunique"),
            average_invoice_nzd=("net_amount",  "mean"),
            max_invoice_nzd   =("net_amount",   "max"),
            min_invoice_nzd   =("net_amount",   "min"),
        ).reset_index().rename(columns={"date": "date"}).merge(df_sales[["store_id", "store_name"]], on="store_id", how="left").sort_values(["date", "store_id"])[["date", "store_id", "store_name",
          "total_sales_nzd", "unique_customers",
          "average_invoice_nzd", "max_invoice_nzd", "min_invoice_nzd"]]

,date,store_id,store_name,total_sales_nzd,unique_customers,average_invoice_nzd,max_invoice_nzd,min_invoice_nzd
0,2026-04-01,akl-001,Auckland Central,172.5,2,86.25,115.0,57.5
1,2026-04-01,wlg-001,Wellington Lambton Quay,92.0,1,92.0,92.0,92.0
2,2026-04-02,chc-001,Christchurch Riccarton,46.0,1,46.0,46.0,46.0
3,2026-04-02,wlg-001,Wellington Lambton Quay,172.5,2,86.25,138.0,34.5
